Aquest notebook forma part dels continguts teòrics dels problemes de l'assignatura d'<strong>Algorísmica</strong> del Grau d'Enginyeria Informàtica a la <a href="https://mat.ub.edu">Facultat de Matemàtiques i Informàtica</a> de la <a href="https://www.ub.edu">Universitat de Barcelona</a></p>Els problemes s'ofereixen sota llicència <a href="https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode">CC-BY-NC-ND license</a>, i el codi sota <a href="https://opensource.org/licenses/MIT">Llicència MIT</a>.</p>

# Visualització interactiva de l'algorisme QuickSort

Aquest notebook permet veure pas a pas com funciona l'algorisme de QuickSort utilitzant visualitzacions amb barres de colors.

In [49]:
# Executeu aquesta cel·la només si cal instal·lar ipywidgets.
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display
import copy


In [79]:
# Estats necessaris per dibuixar cada pas de l'algorisme.
quick_sort_steps = []
sorted_positions = set()

def save_step(
    values,
    pivot_index,
    bounds,
    left_pointer,
    right_pointer,
    action,
):
    """Desa una captura immutable de l'estat actual."""
    quick_sort_steps.append(
        (
            values.copy(),
            pivot_index,
            bounds,
            left_pointer,
            right_pointer,
            action,
            frozenset(sorted_positions),
        )
    )

def mark_singleton_as_sorted(index, values):
    """Incorpora una subllista unitària al pas visual anterior."""
    sorted_positions.add(index)
    if quick_sort_steps:
        previous_step = quick_sort_steps[-1]
        quick_sort_steps[-1] = (
            *previous_step[:-1],
            frozenset(sorted_positions),
        )
    else:
        save_step(values, index, (index, index), -1, -1, "Element ordenat")

def trace_quick_sort(values):
    """Ordena una còpia de ``values`` i en desa els passos visuals."""
    values = values.copy()
    quick_sort_steps.clear()
    sorted_positions.clear()
    quick_sort_traced(values, 0, len(values) - 1)
    return quick_sort_steps

def quick_sort_traced(values, left, right):
    if left > right:
        return
    if left == right:
        mark_singleton_as_sorted(left, values)
        return

    pivot_index = partition_traced(values, left, right)
    quick_sort_traced(values, left, pivot_index - 1)
    quick_sort_traced(values, pivot_index + 1, right)

def partition_traced(values, left, right):
    """Particiona ``values[left:right + 1]`` i en desa cada acció."""
    middle = (left + right) // 2

    # Pivot de la mediana de tres.
    if values[left] > values[middle]:
        values[left], values[middle] = values[middle], values[left]
    if values[left] > values[right]:
        values[left], values[right] = values[right], values[left]
    if values[middle] > values[right]:
        values[middle], values[right] = values[right], values[middle]
    values[middle], values[left] = values[left], values[middle]

    pivot_index = left
    left_pointer, right_pointer = left + 1, right
    bounds = (left, right)

    save_step(
        values,
        pivot_index,
        bounds,
        left_pointer,
        right_pointer,
        "Pivot seleccionat",
    )

    while True:
        while (
            left_pointer <= right
            and values[left_pointer] <= values[pivot_index]
        ):
            left_pointer += 1
            save_step(
                values,
                pivot_index,
                bounds,
                left_pointer,
                right_pointer,
                "i avança",
            )
        while (
            right_pointer >= left
            and values[right_pointer] > values[pivot_index]
        ):
            right_pointer -= 1
            save_step(
                values,
                pivot_index,
                bounds,
                left_pointer,
                right_pointer,
                "j retrocedeix",
            )
        if left_pointer >= right_pointer:
            break

        save_step(
            values,
            pivot_index,
            bounds,
            left_pointer,
            right_pointer,
            "Preparem l'intercanvi entre i i j",
        )
        values[left_pointer], values[right_pointer] = (
            values[right_pointer],
            values[left_pointer],
        )
        save_step(
            values,
            pivot_index,
            bounds,
            left_pointer,
            right_pointer,
            "Intercanviem i amb j",
        )
        left_pointer += 1
        save_step(
            values,
            pivot_index,
            bounds,
            left_pointer,
            right_pointer,
            "i avança",
        )
        right_pointer -= 1
        save_step(
            values,
            pivot_index,
            bounds,
            left_pointer,
            right_pointer,
            "j retrocedeix",
        )

    save_step(
        values,
        pivot_index,
        bounds,
        left_pointer,
        right_pointer,
        "Movem el pivot a la seva posició final",
    )
    values[right_pointer], values[pivot_index] = (
        values[pivot_index],
        values[right_pointer],
    )
    sorted_positions.add(right_pointer)
    save_step(
        values,
        right_pointer,
        bounds,
        left_pointer,
        right_pointer,
        "Pivot a la seva posició final",
    )
    return right_pointer


In [83]:
# Funció de visualització
def plot_quick_sort_step(step):
    from io import BytesIO

    (
        values,
        pivot_index,
        (left, right),
        left_pointer,
        right_pointer,
        action,
        sorted_indices,
    ) = quick_sort_steps[step]
    figure = plt.figure(figsize=(5.5, 3.2))
    axis = figure.add_axes([0.10, 0.14, 0.85, 0.40])

    colors = ['lightgray'] * len(values)
    for index in range(left, right + 1):
        colors[index] = 'skyblue'
    colors[pivot_index] = 'orange'
    for index in sorted_indices:
        colors[index] = 'gold'

    bars = axis.bar(
        range(len(values)),
        values,
        color=colors,
        edgecolor='black',
        linewidth=1.5,
    )
    axis.bar_label(bars, label_type='center')

    if 0 <= left_pointer < len(values):
        bars[left_pointer].set_edgecolor('green')
        bars[left_pointer].set_linewidth(3)
    if 0 <= right_pointer < len(values):
        bars[right_pointer].set_edgecolor('red')
        bars[right_pointer].set_linewidth(3)

    legend_handles = [
        Line2D([0], [0], color='green', linewidth=3, label='i'),
        Line2D([0], [0], color='red', linewidth=3, label='j'),
        Patch(
            facecolor='skyblue',
            edgecolor='black',
            label='Part en ordenació',
        ),
        Patch(facecolor='orange', edgecolor='black', label='Pivot'),
        Patch(
            facecolor='lightgray',
            edgecolor='black',
            label='Part pausada',
        ),
        Patch(
            facecolor='gold',
            edgecolor='black',
            label='Part ja ordenada',
        ),
    ]
    figure.legend(
        handles=legend_handles,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.98),
        ncol=2,
        frameon=False,
    )
    axis.set_title(f"QuickSort - Pas {step}: {action}", fontsize=14)
    axis.set_xticks(range(len(values)))
    axis.set_yticks([])

    buffer = BytesIO()
    figure.savefig(buffer, format='png', dpi=100)
    plt.close(figure)
    plot_qs.value = buffer.getvalue()


In [85]:
example_array = [
    42, 7, 91, 18, 63, 2, 75, 29, 54, 10,
    86, 35, 68, 4, 97, 21, 59, 13, 80, 46,
    6, 72, 31, 94, 16, 57, 25, 88, 39, 65,
]
# example_array = [6, 2, 9, 11, 23, 5, 45, 28, 3, 7, 1, 8]
# example_array = []
# example_array = [6, 2, 9]
# example_array = [1, 2, 3, 4]

In [86]:
# Dades i controls de la visualització.
quick_sort_steps = trace_quick_sort(example_array)

if not quick_sort_steps:
    display(widgets.HTML("<b>La llista és buida.</b>"))
else:
    plot_qs = widgets.Image(
        format='png',
        layout=widgets.Layout(width='550px', height='320px'),
    )
    slider_qs = widgets.IntSlider(
        min=0,
        max=len(quick_sort_steps) - 1,
        step=1,
        description="Pas",
    )
    play_qs = widgets.Play(
        min=0,
        max=len(quick_sort_steps) - 1,
        step=1,
        interval=800,
    )
    next_qs = widgets.Button(icon='step-forward', tooltip='Pas següent')
    widgets.jslink((play_qs, 'value'), (slider_qs, 'value'))

    def on_qs_change(change):
        plot_quick_sort_step(change["new"])

    def on_next_qs_click(_):
        slider_qs.value = min(slider_qs.value + 1, slider_qs.max)

    slider_qs.observe(on_qs_change, names="value")
    next_qs.on_click(on_next_qs_click)

    controls = widgets.HBox(
        [next_qs, play_qs, slider_qs],
        layout=widgets.Layout(width='100%', overflow='hidden'),
    )
    plot_quick_sort_step(0)
    display(controls, plot_qs)


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x02&\x00\x00\x01@\x08\x06\x00\x00\x00\x88\x8b\xb7\x9…

Aquesta llista de problemes i solucions ha estat preparada amb l'ajut de ChatGPT 3.5, i revisada i adaptada per Mireia Ribera. I ha estat revisada a l'agost de 2026, amb l'ajut de Copilot.